# 01 数据概览与探索性分析 (EDA)

本 Notebook 完成以下任务：
- 连接数据库，查看各表行数和字段分布
- 基本统计描述与缺失值概览
- 订单时间分布、用户地域分布

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from utils.db_connector import get_engine
engine = get_engine()
print("数据库连接成功!")

## 1.1 各表行数统计

In [ ]:
# 查看数据库中各表的行数
tables = [
    'olist_orders', 'olist_order_items', 'olist_order_payments',
    'olist_order_reviews', 'olist_products', 'olist_customers',
    'olist_sellers', 'olist_geolocation'
]

table_info = []
with engine.connect() as conn:
    for table in tables:
        try:
            count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table}", conn).iloc[0, 0]
            cols = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 1", conn).columns.tolist()
            table_info.append({'表名': table, '行数': count, '列数': len(cols)})
        except Exception as e:
            table_info.append({'表名': table, '行数': f'错误: {e}', '列数': '-'})

table_summary = pd.DataFrame(table_info)
table_summary

## 1.2 各表字段分布与示例数据

In [ ]:
# 查看订单表结构和示例数据
with engine.connect() as conn:
    orders = pd.read_sql_query("SELECT * FROM olist_orders LIMIT 5", conn)
print("olist_orders 表前5行：")
orders

In [ ]:
# 查看订单表字段类型
with engine.connect() as conn:
    orders_full = pd.read_sql_query("SELECT * FROM olist_orders", conn)
print(f"订单表形状: {orders_full.shape}")
print("\n字段类型：")
print(orders_full.dtypes)
print("\n基本统计描述：")
orders_full.describe(include='all')

## 1.3 缺失值概览

In [ ]:
# 各表缺失值统计
with engine.connect() as conn:
    orders = pd.read_sql_query("SELECT * FROM olist_orders", conn)
    items = pd.read_sql_query("SELECT * FROM olist_order_items", conn)
    reviews = pd.read_sql_query("SELECT * FROM olist_order_reviews", conn)

print("=== 订单表缺失值 ===")
print(orders.isnull().sum())
print(f"\n=== 订单明细表缺失值 ===")
print(items.isnull().sum())
print(f"\n=== 评论表缺失值 ===")
print(reviews.isnull().sum())

## 1.4 订单时间分布

In [ ]:
# 订单时间分布
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['year_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

monthly_orders = orders.groupby('year_month').size()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(monthly_orders)), monthly_orders.values, marker='o', linewidth=2, color='#3498db')
ax.set_xticks(range(0, len(monthly_orders), 2))
ax.set_xticklabels([str(m) for m in monthly_orders.index[::2]], rotation=45, fontsize=8)
ax.set_xlabel('月份')
ax.set_ylabel('订单数')
ax.set_title('月度订单量趋势')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.5 用户地域分布

In [ ]:
# 用户地域分布
with engine.connect() as conn:
    customers = pd.read_sql_query("SELECT * FROM olist_customers", conn)

state_dist = customers['customer_state'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 柱状图
axes[0].bar(state_dist.index[:15], state_dist.values[:15], color='#9b59b6', alpha=0.8)
axes[0].set_xlabel('州')
axes[0].set_ylabel('用户数')
axes[0].set_title('用户分布 Top 15 州')
axes[0].tick_params(axis='x', rotation=45)

# 饼图 Top 5
top5 = state_dist.head(5)
others = pd.Series({'其他': state_dist.iloc[5:].sum()})
pie_data = pd.concat([top5, others])
axes[1].pie(pie_data.values, labels=pie_data.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('用户地域分布占比')

plt.tight_layout()
plt.show()

print(f"\n总用户数: {len(customers)}")
print(f"覆盖州数: {customers['customer_state'].nunique()}")

## 1.6 订单状态分布

In [ ]:
# 订单状态分布
status_dist = orders['order_status'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(status_dist.index, status_dist.values, color='#2ecc71', alpha=0.8)
ax.set_xlabel('订单数')
ax.set_title('订单状态分布')
for i, v in enumerate(status_dist.values):
    ax.text(v + 100, i, str(v), va='center')
plt.tight_layout()
plt.show()

## 小结

- 数据库包含 8 张核心表，覆盖订单、用户、产品、支付、评论等多个维度
- 已送达(delivered)订单占绝大多数，后续分析以此为基础
- 用户主要分布在 SP、RJ、MG 等州
- 订单量在 2017-2018 年呈增长趋势